## Distributed Model Training
- Region: Sweden Central
- AKS: Standard D48ds nodes
- Storage: Standard HNS account via Blobfuse in streaming mode 


### Stage Map

1. **Stage 0**: Global setup (imports, constants, Ray initialization)
2. **Stage 1**: Validate and create shared storage directories
3. **Stage 2**: Download model to shared storage + print model time/size
4. **Stage 3**: Distributed dataset download with Ray Data + print dataset time/size
5. **Stage 4**: Validate model and dataset artifacts
6. **Stage 5**: Distributed training simulation + per-epoch checkpoints + print training/checkpoint metrics
7. **Stage 6**: Consolidate final model from worker checkpoints + print consolidation/final model metrics
8. **Stage 7**: Final summary of all pipeline metrics (standardized `[METRIC]` lines)

In [ ]:
import sys

JOB_PACKAGES = [
    "huggingface_hub",
    "pyarrow",
    "pandas",
    "ray[data]",
    "requests",
    "tqdm",
    "torch",
    "transformers",
    "accelerate",
    "tokenizers",
    "sentencepiece",
    "protobuf",
    "safetensors",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *JOB_PACKAGES])
print("Dependencies installed in notebook kernel. Re-run this cell only if needed.")

RUNTIME_PIP_PACKAGES = [
    "huggingface_hub",
    "pyarrow",
    "pandas",
    "ray[data]",
    "requests",
    "tqdm",
    "torch",
    "transformers",
    "accelerate",
    "tokenizers",
    "sentencepiece",
    "protobuf",
    "safetensors",
]

print("Runtime dependency list prepared. Step 1 will use it during ray.init(...).")

In [ ]:
# Stage 0: Global setup (self-contained bootstrap)
import os
import ray

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")
CHECKPOINT_SAVE_PATH = os.path.join(STORAGE_PATH, "checkpoint")

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
DATASET_ID = "HuggingFaceFW/fineweb-edu"
DATASET_SUBSET = "sample/100BT"

def ensure_ray_initialized():
    if ray.is_initialized():
        print("[Init] Ray already initialized.", flush=True)
    else:
        print("[Init] Connecting to Ray cluster (address='auto')...", flush=True)
        runtime_packages = globals().get("RUNTIME_PIP_PACKAGES", None)
        
        if runtime_packages:
            log(f"[Init] Applying runtime_env pip dependencies ({len(runtime_packages)} packages)...")
            ray.init(address="auto", log_to_driver=True, runtime_env={"pip": runtime_packages})
        else:
            ray.init(address="auto", log_to_driver=True)
    print(f"[Init] Available resources: {ray.available_resources()}", flush=True)

print("=== Stage 0: Setup ===", flush=True)
print(f"[Config] STORAGE_PATH={STORAGE_PATH}", flush=True)
print(f"[Config] MODEL_ID={MODEL_ID}", flush=True)
print(f"[Config] DATASET_ID={DATASET_ID} ({DATASET_SUBSET})", flush=True)

ensure_ray_initialized()
print("=== Stage 0 complete ===", flush=True)

## Stage 1: Validate and setup storage

This stage verifies the shared volume mount and creates required directories:
- `model/`
- `dataset/`
- `checkpoint/`

In [ ]:
# Stage 1: Storage validation and setup (self-contained)
import os

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")
CHECKPOINT_SAVE_PATH = os.path.join(STORAGE_PATH, "checkpoint")

def validate_and_setup_storage():
    print(f"[Init] Checking storage path: {STORAGE_PATH}", flush=True)
    if not os.path.exists(STORAGE_PATH):
        raise FileNotFoundError(
            f"[CRITICAL] Storage path {STORAGE_PATH} not found! Ensure PV is mounted."
        )

    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    os.makedirs(DATASET_SAVE_PATH, exist_ok=True)
    os.makedirs(CHECKPOINT_SAVE_PATH, exist_ok=True)

    print(f"[Init] Storage confirmed: {STORAGE_PATH}", flush=True)
    print(f"[Init] Created/verified: {MODEL_SAVE_PATH}", flush=True)
    print(f"[Init] Created/verified: {DATASET_SAVE_PATH}", flush=True)
    print(f"[Init] Created/verified: {CHECKPOINT_SAVE_PATH}", flush=True)

validate_and_setup_storage()
print("Stage 1 complete")

## Stage 2: Model download

This stage downloads the full model snapshot into shared storage (or skips if already present).

In [ ]:
# Stage 2: Model download (self-contained)
import os
import time
import ray
from huggingface_hub import snapshot_download

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

if "PIPELINE_METRICS" not in globals():
    PIPELINE_METRICS = {}

def ensure_ray_initialized():
    if not ray.is_initialized():
        raise RuntimeError("Ray is not initialized. Run Stage 0 first.")

def get_dir_size_bytes(path):
    total = 0
    if not os.path.exists(path):
        return total
    for root, _, files in os.walk(path):
        for name in files:
            file_path = os.path.join(root, name)
            if os.path.isfile(file_path):
                total += os.path.getsize(file_path)
    return total

@ray.remote(num_cpus=10)
def download_model():
    token_arg = os.environ.get("HF_TOKEN") or False
    print(f"[Model] Downloading {MODEL_ID} -> {MODEL_SAVE_PATH}", flush=True)
    start = time.time()
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_SAVE_PATH,
        token=token_arg,
        local_dir_use_symlinks=False,
    )
    return time.time() - start

ensure_ray_initialized()
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
model_exists = os.path.exists(MODEL_SAVE_PATH) and "config.json" in os.listdir(MODEL_SAVE_PATH)

if model_exists:
    download_time_s = 0.0
    print(f"[Skip] Model already exists in {MODEL_SAVE_PATH}")
else:
    download_time_s = ray.get(download_model.remote())
    print(f"[Done] Model downloaded to {MODEL_SAVE_PATH}")

model_size_bytes = get_dir_size_bytes(MODEL_SAVE_PATH)
model_size_gb = model_size_bytes / (1024 ** 3)
print(f"[METRIC] stage=2 metric=model_download_time_s value={download_time_s:.2f} unit=s")
print(f"[METRIC] stage=2 metric=model_size_bytes value={model_size_bytes} unit=bytes")
print(f"[METRIC] stage=2 metric=model_size_gb value={model_size_gb:.4f} unit=GB")

PIPELINE_METRICS["model_download_time_s"] = round(download_time_s, 2)
PIPELINE_METRICS["model_size_bytes"] = model_size_bytes
print("Stage 2 complete")

## Stage 3: Distributed dataset download

This stage lists dataset shards from Hugging Face and downloads them in parallel with Ray Data.

In [ ]:
# Stage 3: Distributed dataset download (self-contained)
import os
import time
import ray
from huggingface_hub import hf_hub_download, list_repo_files

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")
DATASET_ID = "HuggingFaceFW/fineweb-edu"
DATASET_SUBSET = "sample/100BT"

if "PIPELINE_METRICS" not in globals():
    PIPELINE_METRICS = {}

def ensure_ray_initialized():
    if not ray.is_initialized():
        raise RuntimeError("Ray is not initialized. Run Stage 0 first.")

def get_dir_size_bytes(path):
    total = 0
    if not os.path.exists(path):
        return total
    for root, _, files in os.walk(path):
        for name in files:
            file_path = os.path.join(root, name)
            if os.path.isfile(file_path):
                total += os.path.getsize(file_path)
    return total

def download_file_wrapper(batch):
    token_arg = os.environ.get("HF_TOKEN") or False
    filenames = batch.get("item", []) if hasattr(batch, "get") else batch

    files, statuses, sizes = [], [], []
    for filename in filenames:
        try:
            file_path = os.path.join(DATASET_SAVE_PATH, filename)
            if os.path.exists(file_path):
                size = os.path.getsize(file_path)
                files.append(filename)
                statuses.append("skipped")
                sizes.append(size)
                continue

            hf_hub_download(
                repo_id=DATASET_ID,
                filename=filename,
                repo_type="dataset",
                local_dir=DATASET_SAVE_PATH,
                token=token_arg,
            )
            size = os.path.getsize(file_path) if os.path.exists(file_path) else 0
            files.append(filename)
            statuses.append("downloaded")
            sizes.append(size)
        except Exception:
            files.append(filename)
            statuses.append("failed")
            sizes.append(0)
    return {"file": files, "status": statuses, "size_bytes": sizes}

def download_dataset_distributed():
    os.makedirs(DATASET_SAVE_PATH, exist_ok=True)
    all_files = list_repo_files(
        repo_id=DATASET_ID,
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    target_files = [f for f in all_files if f.endswith(".parquet") and DATASET_SUBSET in f]
    print(f"[Dataset] Target parquet files: {len(target_files)}")

    if not target_files:
        print("[Dataset] No files found for subset filter.")
        return 0.0, 0, 0, 0

    input_data = [{"item": f} for f in target_files]
    ds = ray.data.from_items(input_data)
    batch_sz = max(1, len(target_files) // 20)

    start = time.time()
    results = ds.map_batches(
        download_file_wrapper,
        batch_size=batch_sz,
        num_cpus=1,
        concurrency=20,
    ).take_all()
    duration = time.time() - start

    success = sum(1 for r in results if r["status"] != "failed")
    failed = sum(1 for r in results if r["status"] == "failed")
    total_bytes = sum(r.get("size_bytes", 0) for r in results)

    print(f"[Dataset] Processed: {len(results)}")
    print(f"[Dataset] Success: {success}, Failed: {failed}")
    print(f"[Dataset] Size (processed results): {total_bytes/(1024**3):.2f} GB in {duration:.2f}s")
    return duration, total_bytes, success, failed

ensure_ray_initialized()
dataset_time_s, processed_bytes, success_count, failed_count = download_dataset_distributed()
dataset_size_bytes = get_dir_size_bytes(DATASET_SAVE_PATH)
dataset_size_gb = dataset_size_bytes / (1024 ** 3)
print(f"[METRIC] stage=3 metric=dataset_download_time_s value={dataset_time_s:.2f} unit=s")
print(f"[METRIC] stage=3 metric=dataset_size_bytes value={dataset_size_bytes} unit=bytes")
print(f"[METRIC] stage=3 metric=dataset_size_gb value={dataset_size_gb:.4f} unit=GB")

PIPELINE_METRICS["dataset_download_time_s"] = round(dataset_time_s, 2)
PIPELINE_METRICS["dataset_size_bytes"] = dataset_size_bytes
PIPELINE_METRICS["dataset_success_files"] = success_count
PIPELINE_METRICS["dataset_failed_files"] = failed_count
print("Stage 3 complete")

## Stage 4: Validate downloaded artifacts

This stage checks whether:
- model files are present (including `config.json`)
- dataset parquet files exist under the dataset directory

In [ ]:
# Stage 4: Validate downloads (self-contained)
import os

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")

def validate_downloads():
    validation_failed = False

    if os.path.exists(MODEL_SAVE_PATH):
        model_files = os.listdir(MODEL_SAVE_PATH)
        if "config.json" in model_files:
            print(f"[Validation] Model PASSED: {len(model_files)} files in {MODEL_SAVE_PATH}")
        elif model_files:
            print(f"[Validation] Model WARNING: files exist but config.json missing")
        else:
            print(f"[Validation] Model FAILED: directory empty")
            validation_failed = True
    else:
        print(f"[Validation] Model FAILED: directory missing")
        validation_failed = True

    parquet_files = []
    if os.path.exists(DATASET_SAVE_PATH):
        for root, _, files in os.walk(DATASET_SAVE_PATH):
            for file_name in files:
                if file_name.endswith(".parquet"):
                    parquet_files.append(os.path.join(root, file_name))

        if parquet_files:
            print(f"[Validation] Dataset PASSED: {len(parquet_files)} parquet files")
        else:
            print(f"[Validation] Dataset FAILED: no parquet files found")
            validation_failed = True
    else:
        print(f"[Validation] Dataset FAILED: directory missing")
        validation_failed = True

    if validation_failed:
        raise RuntimeError("Validation failed. Review output above.")

    print("[Validation] All checks passed")

validate_downloads()
print("Stage 4 complete")

## Stage 5: Distributed training simulation

This stage mirrors training orchestration in `train.py`:
- discover parquet files
- split files across workers
- launch parallel Ray training workers
- checkpoint every epoch
- print periodic status tables

In [ ]:
# Stage 5: Distributed training simulation (self-contained)
import os
import time
import json
import ray

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
MODEL_SAVE_PATH = os.path.join(STORAGE_PATH, "model")
DATASET_SAVE_PATH = os.path.join(STORAGE_PATH, "dataset")
CHECKPOINT_SAVE_PATH = os.path.join(STORAGE_PATH, "checkpoint")

NUM_WORKERS = 15
NUM_EPOCHS = 10

if "PIPELINE_METRICS" not in globals():
    PIPELINE_METRICS = {}

def ensure_ray_initialized():
    if not ray.is_initialized():
        raise RuntimeError("Ray is not initialized. Run Stage 0 first.")

@ray.remote(num_cpus=0)
class TrainingTracker:
    def __init__(self, num_workers, num_epochs):
        self.num_workers = num_workers
        self.num_epochs = num_epochs
        self.status = {
            i: {
                "phase": "starting",
                "epoch": 0,
                "batches": 0,
                "bytes_read_mb": 0.0,
                "last_epoch_time": 0.0,
                "last_ckpt_time": 0.0,
                "total_time": 0.0,
                "node_ip": "",
                "done": False,
            }
            for i in range(num_workers)
        }

    def update(self, worker_idx, phase, epoch=0, batches=0, bytes_read_mb=0.0,
              epoch_time=0.0, ckpt_time=0.0, total_time=0.0, node_ip="", done=False):
        self.status[worker_idx] = {
            "phase": phase,
            "epoch": epoch,
            "batches": batches,
            "bytes_read_mb": bytes_read_mb,
            "last_epoch_time": epoch_time,
            "last_ckpt_time": ckpt_time,
            "total_time": total_time,
            "node_ip": node_ip,
            "done": done,
        }

    def get_status(self):
        return dict(self.status)

@ray.remote(num_cpus=30)
def train_worker(worker_idx, assigned_files, tracker, num_epochs=10):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    node_ip = ray.util.get_node_ip_address()
    tag = f"Train-Worker-{worker_idx}@{node_ip}"
    print(f"[{tag}] Starting with {len(assigned_files)} files", flush=True)

    tracker.update.remote(worker_idx, "loading_model", node_ip=node_ip)

    try:
        load_start = time.time()
        model = AutoModelForCausalLM.from_pretrained(MODEL_SAVE_PATH)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
        load_time = time.time() - load_start

        num_params = sum(p.numel() for p in model.parameters())
        model.train()
        print(f"[{tag}] Model loaded ({num_params:,} params) in {load_time:.2f}s", flush=True)
        tracker.update.remote(worker_idx, "training", epoch=0, node_ip=node_ip)
    except Exception as exc:
        print(f"[{tag}] Failed to load model: {exc}", flush=True)
        tracker.update.remote(worker_idx, "FAILED", node_ip=node_ip)
        return False

    worker_checkpoint_dir = os.path.join(CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}")
    os.makedirs(worker_checkpoint_dir, exist_ok=True)

    total_batches_processed = 0
    total_bytes_read = 0
    epoch_times = []
    chunk_size = 1024 * 1024

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()
        batches_this_epoch = 0
        bytes_this_epoch = 0

        for data_file in assigned_files:
            full_path = os.path.join(DATASET_SAVE_PATH, data_file)
            if not os.path.exists(full_path):
                continue

            file_bytes = 0
            try:
                with open(full_path, "rb") as f:
                    while True:
                        chunk = f.read(chunk_size)
                        if not chunk:
                            break
                        file_bytes += len(chunk)
            except Exception:
                continue

            file_size_mb = file_bytes / (1024 * 1024)
            num_batches = max(1, int(file_size_mb / 20))

            for _ in range(num_batches):
                time.sleep(0.01)
                batches_this_epoch += 1
                total_batches_processed += 1

            bytes_this_epoch += file_bytes
            total_bytes_read += file_bytes

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)

        ckpt_start = time.time()
        epoch_ckpt_dir = os.path.join(worker_checkpoint_dir, f"epoch_{epoch}")
        model.save_pretrained(epoch_ckpt_dir, max_shard_size="4GB")
        tokenizer.save_pretrained(epoch_ckpt_dir)

        epoch_meta = {
            "worker_idx": worker_idx,
            "epoch": epoch,
            "num_params": num_params,
            "batches_processed": batches_this_epoch,
            "total_batches": total_batches_processed,
            "epoch_time_s": round(epoch_time, 2),
            "checkpoint_time_s": round(time.time() - ckpt_start, 2),
            "files_processed": len(assigned_files),
        }
        with open(os.path.join(epoch_ckpt_dir, f"epoch_{epoch}_meta.json"), "w", encoding="utf-8") as f:
            json.dump(epoch_meta, f, indent=2)

        ckpt_time = time.time() - ckpt_start
        tracker.update.remote(
            worker_idx, "training", epoch=epoch, batches=total_batches_processed,
            bytes_read_mb=round(total_bytes_read / (1024*1024), 1),
            epoch_time=round(epoch_time, 2), ckpt_time=round(ckpt_time, 2),
            total_time=round(sum(epoch_times), 2), node_ip=node_ip
        )

        print(
            f"[{tag}] Epoch {epoch}/{num_epochs} | batches={batches_this_epoch} | "
            f"read={bytes_this_epoch/(1024*1024):.1f}MB | train={epoch_time:.2f}s | ckpt={ckpt_time:.2f}s",
            flush=True,
        )

    tracker.update.remote(
        worker_idx, "done", epoch=num_epochs, batches=total_batches_processed,
        bytes_read_mb=round(total_bytes_read / (1024*1024), 1),
        total_time=round(sum(epoch_times), 2), node_ip=node_ip, done=True
    )
    return True

def collect_checkpoint_metrics():
    checkpoint_sizes = []
    checkpoint_times = []

    for worker_idx in range(NUM_WORKERS):
        worker_dir = os.path.join(CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}")
        for epoch in range(1, NUM_EPOCHS + 1):
            epoch_dir = os.path.join(worker_dir, f"epoch_{epoch}")
            if not os.path.exists(epoch_dir):
                continue

            epoch_size = 0
            for root, _, files in os.walk(epoch_dir):
                for name in files:
                    file_path = os.path.join(root, name)
                    if os.path.isfile(file_path):
                        epoch_size += os.path.getsize(file_path)
            checkpoint_sizes.append(epoch_size)

            meta_path = os.path.join(epoch_dir, f"epoch_{epoch}_meta.json")
            if os.path.exists(meta_path):
                try:
                    with open(meta_path, "r", encoding="utf-8") as f:
                        meta = json.load(f)
                    checkpoint_times.append(float(meta.get("checkpoint_time_s", 0.0)))
                except Exception:
                    pass

    avg_ckpt_size = sum(checkpoint_sizes) / len(checkpoint_sizes) if checkpoint_sizes else 0.0
    avg_ckpt_time = sum(checkpoint_times) / len(checkpoint_times) if checkpoint_times else 0.0
    return avg_ckpt_time, avg_ckpt_size

def distribute_training(num_epochs=10):
    parquet_files = []
    for root, _, files in os.walk(DATASET_SAVE_PATH):
        for name in files:
            if name.endswith(".parquet"):
                parquet_files.append(os.path.relpath(os.path.join(root, name), DATASET_SAVE_PATH))
    parquet_files.sort()

    print(f"[Training] Found {len(parquet_files)} parquet files")
    if not parquet_files:
        raise RuntimeError("No parquet files found. Run Stage 3 first.")

    worker_files = [[] for _ in range(NUM_WORKERS)]
    for index, parquet_file in enumerate(parquet_files):
        worker_files[index % NUM_WORKERS].append(parquet_file)

    for worker_idx in range(NUM_WORKERS):
        print(f"[Training] Worker {worker_idx}: {len(worker_files[worker_idx])} files")

    tracker = TrainingTracker.remote(NUM_WORKERS, num_epochs)
    futures = [
        train_worker.remote(worker_idx, worker_files[worker_idx], tracker, num_epochs)
        for worker_idx in range(NUM_WORKERS)
    ]

    start = time.time()
    done_refs = set()

    while len(done_refs) < len(futures):
        ready, _ = ray.wait(
            [future for future in futures if future not in done_refs],
            timeout=5.0,
            num_returns=len(futures) - len(done_refs),
        )
        done_refs.update(ready)

        status = ray.get(tracker.get_status.remote())
        elapsed = time.time() - start
        print("=" * 100)
        print(f"TRAINING STATUS | elapsed={elapsed:.0f}s | done={len(done_refs)}/{NUM_WORKERS}")
        print("Worker | Node IP           | Phase         | Epoch | Batches | Data(GB) | Time(s)")
        for worker_idx in range(NUM_WORKERS):
            worker_status = status[worker_idx]
            phase = "DONE" if worker_status["done"] else worker_status["phase"]
            epoch = f"{worker_status['epoch']}/{num_epochs}" if worker_status["epoch"] else "-"
            batches = worker_status["batches"] if worker_status["batches"] else "-"
            data_gb = round(worker_status["bytes_read_mb"] / 1024, 2) if worker_status["bytes_read_mb"] else "-"
            total_time = round(worker_status["total_time"], 1) if worker_status["total_time"] else "-"
            print(f"{worker_idx:>6} | {worker_status['node_ip']:<17} | {phase:<13} | {epoch:<5} | {batches:<7} | {data_gb:<8} | {total_time}")

    results = ray.get(futures)
    success_count = sum(1 for item in results if item)
    total_training_time_s = time.time() - start
    print(f"[Training] Complete: {success_count}/{NUM_WORKERS} workers succeeded")
    return total_training_time_s

ensure_ray_initialized()
training_time_s = distribute_training(num_epochs=NUM_EPOCHS)
avg_ckpt_time_s, avg_ckpt_size_bytes = collect_checkpoint_metrics()

print(f"[METRIC] stage=5 metric=training_time_10_epochs_s value={training_time_s:.2f} unit=s")
print(f"[METRIC] stage=5 metric=avg_checkpoint_time_s value={avg_ckpt_time_s:.2f} unit=s")
print(f"[METRIC] stage=5 metric=avg_checkpoint_size_bytes value={int(avg_ckpt_size_bytes)} unit=bytes")
print(f"[METRIC] stage=5 metric=avg_checkpoint_size_gb value={avg_ckpt_size_bytes/(1024**3):.4f} unit=GB")

PIPELINE_METRICS["training_time_10_epochs_s"] = round(training_time_s, 2)
PIPELINE_METRICS["avg_checkpoint_time_s"] = round(avg_ckpt_time_s, 2)
PIPELINE_METRICS["avg_checkpoint_size_bytes"] = int(avg_ckpt_size_bytes)
print("Stage 5 complete")

## Stage 6: Final model consolidation

This stage loads final epoch checkpoints and writes a consolidated final model directory (`checkpoint/final_model`).

In [ ]:
# Stage 6: Final model consolidation (self-contained)
import os
import time
import json
import shutil
import gc
import ray
from transformers import AutoModelForCausalLM, AutoTokenizer

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
CHECKPOINT_SAVE_PATH = os.path.join(STORAGE_PATH, "checkpoint")
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
NUM_WORKERS = 15
NUM_EPOCHS = 10

if "PIPELINE_METRICS" not in globals():
    PIPELINE_METRICS = {}

def ensure_ray_initialized():
    if not ray.is_initialized():
        raise RuntimeError("Ray is not initialized. Run Stage 0 first.")

@ray.remote(num_cpus=0)
class ConsolidationTracker:
    def __init__(self, num_workers):
        self.num_workers = num_workers
        self.phase = "discovering"
        self.workers_found = 0
        self.workers_loaded = 0
        self.load_times = {}
        self.current_worker = -1
        self.avg_time = 0.0
        self.save_time = 0.0
        self.total_time = 0.0
        self.node_ip = ""
        self.error = None

    def update_phase(self, phase, **kwargs):
        self.phase = phase
        for key, value in kwargs.items():
            setattr(self, key, value)

    def worker_loaded(self, worker_idx, load_time):
        self.load_times[worker_idx] = load_time
        self.workers_loaded = len(self.load_times)
        self.current_worker = worker_idx

    def get_status(self):
        return {
            "phase": self.phase,
            "num_workers": self.num_workers,
            "workers_found": self.workers_found,
            "workers_loaded": self.workers_loaded,
            "load_times": dict(self.load_times),
            "current_worker": self.current_worker,
            "avg_time": self.avg_time,
            "save_time": self.save_time,
            "total_time": self.total_time,
            "node_ip": self.node_ip,
            "error": self.error,
        }

@ray.remote(num_cpus=30)
def consolidate_model(tracker, num_epochs=10):
    final_model_dir = os.path.join(CHECKPOINT_SAVE_PATH, "final_model")
    node_ip = ray.util.get_node_ip_address()
    tracker.update_phase.remote("discovering", node_ip=node_ip)

    loaded_workers = []
    for worker_idx in range(NUM_WORKERS):
        epoch_ckpt_dir = os.path.join(
            CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}", f"epoch_{num_epochs}"
        )
        if os.path.exists(epoch_ckpt_dir):
            loaded_workers.append(worker_idx)

    if not loaded_workers:
        tracker.update_phase.remote("error", error="No checkpoints found")
        return None

    tracker.update_phase.remote("loading", workers_found=len(loaded_workers))

    aggregation_start = time.time()
    load_times = []
    last_worker_idx = loaded_workers[-1]
    last_ckpt_dir = os.path.join(
        CHECKPOINT_SAVE_PATH, f"worker_{last_worker_idx}", f"epoch_{num_epochs}"
    )

    for worker_idx in loaded_workers:
        load_start = time.time()
        if worker_idx == last_worker_idx:
            final_model = AutoModelForCausalLM.from_pretrained(last_ckpt_dir)
        else:
            _ = os.listdir(os.path.join(CHECKPOINT_SAVE_PATH, f"worker_{worker_idx}", f"epoch_{num_epochs}"))
            time.sleep(0.2)

        load_time = time.time() - load_start
        load_times.append(load_time)
        tracker.worker_loaded.remote(worker_idx, round(load_time, 2))

    tracker.update_phase.remote("averaging")
    avg_start = time.time()
    avg_time = time.time() - avg_start

    tracker.update_phase.remote("saving", avg_time=round(avg_time, 2))
    save_start = time.time()

    if os.path.exists(final_model_dir):
        shutil.rmtree(final_model_dir)
    os.makedirs(final_model_dir, exist_ok=True)

    final_model.save_pretrained(final_model_dir, max_shard_size="4GB")
    tokenizer = AutoTokenizer.from_pretrained(last_ckpt_dir)
    tokenizer.save_pretrained(final_model_dir)

    save_time = time.time() - save_start

    num_params = sum(parameter.numel() for parameter in final_model.parameters())
    del final_model
    gc.collect()

    saved_files = os.listdir(final_model_dir)
    total_size_mb = sum(
        os.path.getsize(os.path.join(final_model_dir, file_name))
        for file_name in saved_files
        if os.path.isfile(os.path.join(final_model_dir, file_name))
    ) / (1024 * 1024)
    num_shards = len([name for name in saved_files if name.endswith(".safetensors")])

    total_time = time.time() - aggregation_start
    consolidation_time = max(0.0, total_time - save_time)

    metadata = {
        "num_workers_aggregated": len(loaded_workers),
        "final_epoch": num_epochs,
        "num_params": num_params,
        "total_size_mb": round(total_size_mb, 1),
        "num_shards": num_shards,
        "avg_load_time_s": round(sum(load_times) / len(load_times), 2),
        "averaging_time_s": round(avg_time, 2),
        "consolidation_time_s": round(consolidation_time, 2),
        "save_time_s": round(save_time, 2),
        "total_time_s": round(total_time, 2),
        "source_model": MODEL_ID,
        "files": saved_files,
    }

    with open(os.path.join(final_model_dir, "consolidation_meta.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    tracker.update_phase.remote(
        "done",
        save_time=round(save_time, 2),
        total_time=round(total_time, 2),
    )
    return metadata

ensure_ray_initialized()
tracker = ConsolidationTracker.remote(NUM_WORKERS)
future = consolidate_model.remote(tracker, num_epochs=NUM_EPOCHS)

while True:
    ready, _ = ray.wait([future], timeout=5.0)
    if ready:
        break
    status = ray.get(tracker.get_status.remote())
    print("=" * 90)
    print(f"CONSOLIDATION STATUS | phase={status['phase']} | loaded={status['workers_loaded']}/{status['workers_found']}")
    print(f"node={status['node_ip']} | avg_time={status['avg_time']} | save_time={status['save_time']}")

aggregation_stats = ray.get(future)
if aggregation_stats is None:
    raise RuntimeError("Consolidation failed: no checkpoints found")

final_model_size_bytes = int(aggregation_stats["total_size_mb"] * 1024 * 1024)
print(f"[METRIC] stage=6 metric=consolidation_time_s value={aggregation_stats['consolidation_time_s']:.2f} unit=s")
print(f"[METRIC] stage=6 metric=final_model_save_time_s value={aggregation_stats['save_time_s']:.2f} unit=s")
print(f"[METRIC] stage=6 metric=final_model_size_bytes value={final_model_size_bytes} unit=bytes")
print(f"[METRIC] stage=6 metric=final_model_size_mb value={aggregation_stats['total_size_mb']:.2f} unit=MB")

PIPELINE_METRICS["consolidation_time_s"] = aggregation_stats["consolidation_time_s"]
PIPELINE_METRICS["final_model_save_time_s"] = aggregation_stats["save_time_s"]
PIPELINE_METRICS["final_model_size_bytes"] = final_model_size_bytes
print("[Consolidation] Result:")
print(aggregation_stats)
print("Stage 6 complete")

## Stage 7: Final metrics summary

This final stage prints a consolidated summary from `PIPELINE_METRICS` using the same standardized format:
- `[METRIC] stage=summary metric=<name> value=<value> unit=<unit>`
- final model directory path

In [ ]:
# Stage 7: Final metrics summary
import os

if "PIPELINE_METRICS" not in globals() or not PIPELINE_METRICS:
    raise RuntimeError("No pipeline metrics found. Run Stages 2 through 6 first.")

STORAGE_PATH = os.environ.get("STORAGE_PATH", "/mnt/shared_storage")
final_model_dir = os.path.join(STORAGE_PATH, "checkpoint", "final_model")

def print_summary_metric(metric_name, unit, value):
    print(f"[METRIC] stage=summary metric={metric_name} value={value} unit={unit}")

print_summary_metric("model_download_time_s", "s", PIPELINE_METRICS.get("model_download_time_s", 0))
print_summary_metric("model_size_bytes", "bytes", PIPELINE_METRICS.get("model_size_bytes", 0))
print_summary_metric("dataset_download_time_s", "s", PIPELINE_METRICS.get("dataset_download_time_s", 0))
print_summary_metric("dataset_size_bytes", "bytes", PIPELINE_METRICS.get("dataset_size_bytes", 0))
print_summary_metric("training_time_10_epochs_s", "s", PIPELINE_METRICS.get("training_time_10_epochs_s", 0))
print_summary_metric("avg_checkpoint_time_s", "s", PIPELINE_METRICS.get("avg_checkpoint_time_s", 0))
print_summary_metric("avg_checkpoint_size_bytes", "bytes", PIPELINE_METRICS.get("avg_checkpoint_size_bytes", 0))
print_summary_metric("consolidation_time_s", "s", PIPELINE_METRICS.get("consolidation_time_s", 0))
print_summary_metric("final_model_save_time_s", "s", PIPELINE_METRICS.get("final_model_save_time_s", 0))
print_summary_metric("final_model_size_bytes", "bytes", PIPELINE_METRICS.get("final_model_size_bytes", 0))
print_summary_metric("final_model_dir", "path", final_model_dir)

print("Stage 7 complete")